# 10 — Propagação da riqueza W_{t+1}=S_t·R_p

Desenvolve `propagar_riqueza`. **F10.**

In [ ]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np

setup OK


## Desenvolvimento

A função abaixo foi escrita aqui e, após os testes, movida para `app/nucleo.py`.

In [ ]:
def propagar_riqueza(w0, theta, alpha, R, rf):
    """Forward pass (Etapas 5–6): consome c_t=θ_t·W_t, investe a poupança em α*
    e propaga W_{t+1}=S_t·R_p. (F9, F10)

    Parameters
    ----------
    w0 : riqueza inicial W₀.
    theta : (T+1,) frações de consumo θ_t.
    alpha : (N,) carteira ótima α*.
    R : (n_paths, T, N) retornos brutos amostrados (um por período e caminho).
    rf : fator livre de risco bruto.

    Returns
    -------
    dict com ``W``, ``c``, ``S`` — cada um shape (n_paths, T+1).
    """
    R = np.asarray(R, dtype=float)
    alpha = np.asarray(alpha, dtype=float)
    theta = np.asarray(theta, dtype=float)
    n_paths, T, _ = R.shape
    W = np.empty((n_paths, T + 1))
    c = np.empty((n_paths, T + 1))
    S = np.empty((n_paths, T + 1))
    W[:, 0] = w0
    for t in range(T):
        c[:, t] = theta[t] * W[:, t]
        S[:, t] = W[:, t] - c[:, t]
        R_p = rf + (R[:, t, :] - rf) @ alpha           # (n_paths,)
        W[:, t + 1] = S[:, t] * R_p
    # Condição terminal: consome toda a riqueza (θ_T = 1).
    c[:, T] = theta[T] * W[:, T]
    S[:, T] = W[:, T] - c[:, T]
    return {"W": W, "c": c, "S": S}


**Teste** — W_{t+1}=S_t·R_p e W ≥ 0.

In [3]:
rng = np.random.default_rng(1); rf_g, T = 1.003, 6
theta = np.linspace(0.1, 1.0, T+1); alpha = np.array([0.5])
Rp = np.maximum(1+rng.normal(0.01,0.05,(5,T,1)),0)
out = propagar_riqueza(1.0, theta, alpha, Rp, rf_g)
print('W:'); print(out['W'])
for t in range(T):
    assert np.allclose(out['W'][:,t+1], out['S'][:,t]*(rf_g+(Rp[:,t,:]-rf_g)@alpha))
assert np.all(out['W']>=0)
print('F10 propagacao: PASSOU')

W:
[[1.         0.91362564 0.70374787 0.42848151 0.18778822 0.05797777
  0.00885024]
 [1.         0.89376855 0.68442201 0.41706527 0.19027935 0.05749541
  0.00879825]
 [1.         0.88927978 0.66857872 0.39891967 0.18336822 0.05542266
  0.00830665]
 [1.         0.88825706 0.66623956 0.40242344 0.18101991 0.05641585
  0.00873036]
 [1.         0.84484884 0.60783157 0.36547601 0.16379734 0.04972106
  0.00754716]]
F10 propagacao: PASSOU
